In [1]:
##Use Annotated when we need to append state messages 
from typing import Annotated
from typing_extensions import TypedDict
from operator import add

class State(TypedDict):
    foo: int
    bar: Annotated[list[str], add]


In [2]:
## We can use cache policy when add node to the graph
from langgraph.types import CachePolicy
from langgraph.cache.memory import InMemoryCache
##builder.add_node("expensive_node", expensive_node, cache_policy=CachePolicy(ttl=3))
##TTL means time to live, which is the duration for which the cache entry is valid.
## ADD InMemoryCache. when you compile the graph
#graph = builder.compile(cache=InMemoryCache())

In [3]:
## Add conditional edges when use kind of routing function to decide which node to go next
##graph.add_conditional_edges("node_a", routing_function)
##You can optionally provide a dictionary that maps the routing_function's output to the name of the next node.
## graph.add_conditional_edges("node_a", routing_function, {True: "node_b", False: "node_c"})

In [ ]:
## You can optionally specify a config_schema when creating a graph.
from langgraph.graph import StateGraph, START, END
class ConfigSchema(TypedDict):
    llm: str

graph = StateGraph(State, config_schema=ConfigSchema)

##You can then pass this configuration into the graph using the configurable config field.
# config = {"configurable": {"llm": "anthropic"}}
# graph.invoke(inputs, config=config)
# You can then access and use this configuration inside a node or conditional edge:
def node_a(state, config):
    llm_type = config.get("configurable", {}).get("llm", "openai")
    llm = get_llm(llm_type)
    ...

##Add runtime configuration for specify model and system message 
## https://langchain-ai.github.io/langgraph/how-tos/graph-api/#add-runtime-configuration 

In [ ]:
##Our node is just a Python function that reads our graph's state and makes updates to it. 
# The first argument to this function will always be the state.
# We kicked off invocation by updating a single key of the state.
# and we can receive the entire state in the invocation result.


In [10]:
'''
By default, StateGraph operates with a single schema, and all nodes are expected to communicate using that schema. However, 
it's also possible to define distinct input and output schemas for a graph.

When distinct schemas are specified, an internal schema will still be used for communication between nodes. 
The input schema ensures that the provided input matches the expected structure, 
while the output schema filters the internal data to return only the relevant information according to the defined output schema.
'''
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict

# Define the schema for the input
class InputState(TypedDict):
    question: str

# Define the schema for the output
class OutputState(TypedDict):
    answer: str

# Define the overall schema, combining both input and output
class OverallState(InputState, OutputState):
    pass

# Define the node that processes the input and generates an answer
def answer_node(state: InputState):
    # Example answer and an extra key
    return {"answer": "bye", "question": state["question"]}

# Build the graph with input and output schemas specified
builder = StateGraph(OverallState, input_schema=InputState, output_schema=OutputState)
builder.add_node(answer_node)  # Add the answer node
builder.add_edge(START, "answer_node")  # Define the starting edge
builder.add_edge("answer_node", END)  # Define the ending edge
graph = builder.compile()  # Compile the graph

# Invoke the graph with an input and print the result
print(graph.invoke({"question": "hi"}))

{'answer': 'bye'}


In [11]:
##Notice that the above output of invoke only includes the output schema.

In [13]:
##Add retry policies
'''
from langgraph.pregel import RetryPolicy

builder.add_node(
    "node_name",
    node_function,
    retry_policy=RetryPolicy(),
)
.....
.....
builder.add_node("model", call_model, retry_policy=RetryPolicy(max_attempts=5))
'''

'\nfrom langgraph.pregel import RetryPolicy\n\nbuilder.add_node(\n    "node_name",\n    node_function,\n    retry_policy=RetryPolicy(),\n)\n.....\n.....\nbuilder.add_node("model", call_model, retry_policy=RetryPolicy(max_attempts=5))\n'

In [17]:
##Create a sequence of steps
from langgraph.graph import START, StateGraph

builder = StateGraph(State)

# Add nodes
#builder.add_node(step_1)
#builder.add_node(step_2)
#builder.add_node(step_3)

# Add edges
#builder.add_edge(START, "step_1")
#builder.add_edge("step_1", "step_2")
#builder.add_edge("step_2", "step_3")

##We can also use the built-in shorthand .add_sequence:


##builder = StateGraph(State).add_sequence([step_1, step_2, step_3])
##builder.add_edge(START, "step_1")

In [28]:
import operator
from typing import Annotated, Literal, Sequence
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    aggregate: Annotated[list, operator.add]
    # Add a key to the state. We will set this key to determine
    # how we branch.
    which: str

def a(state: State):
    print(f'Adding "A" to {state["aggregate"]}')
    return {"aggregate": ["A"], "which": "x"}

def x(state: State):
    print(f'Adding "B" to {state["aggregate"]}')
    return {"aggregate": ["B"]}

def c(state: State):
    print(f'Adding "C" to {state["aggregate"]}')
    return {"aggregate": ["C"]}

builder = StateGraph(State)
builder.add_node(a)
builder.add_node(x)
builder.add_node(c)
builder.add_edge(START, "a")
builder.add_edge("b", END)
builder.add_edge("c", END)

def conditional_edge(state: State) -> Literal["x", "c"]:
    # Fill in arbitrary logic here that uses the state
    # to determine the next node
    return state["which"]

builder.add_conditional_edges("a", conditional_edge)

graph = builder.compile()
result = graph.invoke({"aggregate": []})
print(result)


ValueError: Found edge starting at unknown node 'b'